In [1]:
import pandas as pd
import sqlite3
from datetime import datetime, timedelta
import numpy as np
from loguru import logger
import os

logger.add("etl_proces.log", rotation="5 MB")


1

 # Stap 1 E(Extract)

In [2]:
bron_db_pad = '../week_3/DE/sdm.db'
if not os.path.exists(bron_db_pad):
    logger.warning(f"Bron database niet gevonden op pad: {bron_db_pad}. Controleer het pad!")

logger.info(f"Stap 1: Extract - Verbinden met bron database '{bron_db_pad}'")
bron_conn = sqlite3.connect(bron_db_pad)


2026-04-09 14:17:40.817 | INFO     | __main__:<module>:5 - Stap 1: Extract - Verbinden met bron database '../week_3/DE/sdm.db'


In [3]:
logger.info("Ophalen Inkoop data...")
df_acc_inkoop = pd.read_sql_query("SELECT * FROM Accessoire_Inkoop", bron_conn)
df_acc_inkoop_acc = pd.read_sql_query("SELECT * FROM Accessoire_Inkoop_Accessoire", bron_conn)
df_acc_inkoop_lev = pd.read_sql_query("SELECT * FROM Accessoire_Inkoop_Leverancier", bron_conn)

df_fiets_inkoop = pd.read_sql_query("SELECT * FROM Fiets_Inkoop", bron_conn)
df_fiets_inkoop_fab = pd.read_sql_query("SELECT * FROM Fiets_Inkoop_Fabrikant", bron_conn)
df_fiets_inkoop_fiets = pd.read_sql_query("SELECT * FROM Fiets_Inkoop_Fiets", bron_conn)


2026-04-09 14:17:40.825 | INFO     | __main__:<module>:1 - Ophalen Inkoop data...


In [4]:
logger.info("Ophalen Verkoop data...")
df_acc_verk = pd.read_sql_query("SELECT * FROM Accessoireverkoop", bron_conn)
df_acc_verk_acc = pd.read_sql_query("SELECT * FROM Accessoireverkoop_Accessoire", bron_conn)
df_acc_verk_fil = pd.read_sql_query("SELECT * FROM Accessoireverkoop_filiaal", bron_conn)
df_acc_verk_klant = pd.read_sql_query("SELECT * FROM Accessoireverkoop_klant", bron_conn)
df_acc_verk_lev = pd.read_sql_query("SELECT * FROM Accessoireverkoop_Leverancier", bron_conn)
df_acc_verk_mont = pd.read_sql_query("SELECT * FROM Accessoireverkoop_Monteur", bron_conn)

df_fiets_verk = pd.read_sql_query("SELECT * FROM Fietsverkoop", bron_conn)
df_fiets_verk_fab = pd.read_sql_query("SELECT * FROM Fietsverkoop_Fabrikant", bron_conn)
df_fiets_verk_fiets = pd.read_sql_query("SELECT * FROM Fietsverkoop_Fiets", bron_conn)
df_fiets_verk_fil = pd.read_sql_query("SELECT * FROM Fietsverkoop_filiaal", bron_conn)
df_fiets_verk_klant = pd.read_sql_query("SELECT * FROM Fietsverkoop_klant", bron_conn)
df_fiets_verk_mont = pd.read_sql_query("SELECT * FROM Fietsverkoop_Monteur", bron_conn)


2026-04-09 14:17:40.845 | INFO     | __main__:<module>:1 - Ophalen Verkoop data...


In [5]:
logger.info("Ophalen Onderhoud data...")
df_ond = pd.read_sql_query("SELECT * FROM Onderhoud", bron_conn)
df_ond_fab = pd.read_sql_query("SELECT * FROM Onderhoud_Fabrikant", bron_conn)
df_ond_fiets = pd.read_sql_query("SELECT * FROM Onderhoud_Fiets", bron_conn)
df_ond_fil = pd.read_sql_query("SELECT * FROM Onderhoud_Filiaal", bron_conn)
df_ond_mont = pd.read_sql_query("SELECT * FROM Onderhoud_Monteur", bron_conn)


2026-04-09 14:17:40.869 | INFO     | __main__:<module>:1 - Ophalen Onderhoud data...


In [6]:
bron_conn.close()
logger.success("Stap 1: Extract afgerond. Data is ingeladen in werkgeheugen.")


2026-04-09 14:17:40.883 | SUCCESS  | __main__:<module>:2 - Stap 1: Extract afgerond. Data is ingeladen in werkgeheugen.


 # Stap 2 T (Transform)

In [7]:
logger.info("Stap 2: Transform - Bouwen van Dimensies en Feiten...")

# DIM KLANT
dim_klant = pd.concat([df_acc_verk_klant, df_fiets_verk_klant]).drop_duplicates(
    subset=['klantnr']).reset_index(drop=True)
huidig_jaar = datetime.now().year
dim_klant['geboortedatum'] = pd.to_datetime(
    dim_klant['geboortedatum'], errors='coerce')
dim_klant['leeftijd'] = huidig_jaar - dim_klant['geboortedatum'].dt.year
dim_klant['leeftijdgroep'] = pd.cut(dim_klant['leeftijd'], bins=[
                                    0, 20, 40, 60, 100], labels=['<20', '20-40', '40-60', '60+'])
dim_klant = dim_klant.drop(columns=['leeftijd'])
logger.info(f"-> DIM_KLANT getransformeerd: {len(dim_klant)} unieke klanten.")


2026-04-09 14:17:40.898 | INFO     | __main__:<module>:1 - Stap 2: Transform - Bouwen van Dimensies en Feiten...
2026-04-09 14:17:40.910 | INFO     | __main__:<module>:13 - -> DIM_KLANT getransformeerd: 25 unieke klanten.


In [8]:
# DIM FIETS
dim_fiets_basis = pd.concat([df_fiets_inkoop_fiets, df_fiets_verk_fiets,
                            df_ond_fiets]).drop_duplicates(subset=['fietsnr'])
dim_fabrikant = pd.concat([df_fiets_inkoop_fab, df_fiets_verk_fab,
                          df_ond_fab]).drop_duplicates(subset=['fabrikantnr'])
# Koppel fiets aan fabrikant
dim_fiets = pd.merge(dim_fiets_basis, dim_fabrikant, left_on='fabrikant',
                     right_on='fabrikantnr', how='left', suffixes=('', '_fabrikant'))
dim_fiets = dim_fiets.rename(columns={
                             'naam': 'fabrikantnaam', 'adres': 'fabrikant_adres', 'plaats': 'fabrikant_plaats'})
dim_fiets['prijsklasse'] = np.where(
    dim_fiets['standaardprijs'] > 1000, 'Duur Segment', 'Basis Segment')
logger.info(f"-> DIM_FIETS getransformeerd: {len(dim_fiets)} unieke fietsen.")


2026-04-09 14:17:40.930 | INFO     | __main__:<module>:13 - -> DIM_FIETS getransformeerd: 75 unieke fietsen.


In [9]:
# DIM ACCESSOIRE
dim_acc_basis = pd.concat(
    [df_acc_inkoop_acc, df_acc_verk_acc]).drop_duplicates(subset=['accessoirenr'])
dim_lev = pd.concat([df_acc_inkoop_lev, df_acc_verk_lev]
                    ).drop_duplicates(subset=['leveranciernr'])
# Koppel accessoire aan leverancier
dim_accessoire = pd.merge(dim_acc_basis, dim_lev, left_on='leverancier',
                          right_on='leveranciernr', how='left', suffixes=('', '_leverancier'))
dim_accessoire = dim_accessoire.rename(
    columns={'naam_leverancier': 'leveranciernaam', 'adres': 'leverancier_adres', 'woonplaats': 'leverancier_woonplaats'})
logger.info(f"-> DIM_ACCESSOIRE getransformeerd: {len(dim_accessoire)} unieke accessoires.")


2026-04-09 14:17:40.948 | INFO     | __main__:<module>:11 - -> DIM_ACCESSOIRE getransformeerd: 10 unieke accessoires.


In [10]:
# DIM MONTEUR
dim_monteur_basis = pd.concat(
    [df_acc_verk_mont, df_fiets_verk_mont, df_ond_mont]).drop_duplicates(subset=['monteurnr'])
dim_filiaal = pd.concat([df_acc_verk_fil, df_fiets_verk_fil,
                        df_ond_fil]).drop_duplicates(subset=['filiaalnr'])
# Koppel monteur aan filiaal
dim_monteur = pd.merge(dim_monteur_basis, dim_filiaal, left_on='filiaal',
                       right_on='filiaalnr', how='left', suffixes=('', '_filiaal'))
dim_monteur = dim_monteur.rename(
    columns={'naam_filiaal': 'filiaalnaam', 'adres': 'filiaal_adres'})
dim_monteur['salarisgroep'] = np.where(
    dim_monteur['uurloon'] > 20, 'Senior', 'Medior')
logger.info(f"-> DIM_MONTEUR getransformeerd: {len(dim_monteur)} unieke monteurs.")


2026-04-09 14:17:40.967 | INFO     | __main__:<module>:13 - -> DIM_MONTEUR getransformeerd: 15 unieke monteurs.


In [11]:
# DIM DATUM
dim_datum = pd.DataFrame(
    {'datum': pd.date_range(start='2020-01-01', end='2025-12-31')})
dim_datum['datum_id'] = dim_datum['datum'].dt.strftime('%Y%m%d').astype(int)
dim_datum['dag'] = dim_datum['datum'].dt.day
dim_datum['maand'] = dim_datum['datum'].dt.month
dim_datum['jaar'] = dim_datum['datum'].dt.year
dim_datum['maandnaam'] = dim_datum['datum'].dt.strftime('%B')
dim_datum['maand_id'] = dim_datum['datum'].dt.strftime('%Y%m').astype(int)
logger.info(f"-> DIM_DATUM getransformeerd: {len(dim_datum)} datums gegenereerd.")


2026-04-09 14:17:41.031 | INFO     | __main__:<module>:10 - -> DIM_DATUM getransformeerd: 2192 datums gegenereerd.


In [12]:
# FEIT INKOOP
df_acc_inkoop_feit = df_acc_inkoop.copy()
df_acc_inkoop_feit = df_acc_inkoop_feit.rename(
    columns={'accessoire': 'accessoirenummer'})
df_acc_inkoop_feit['fietsnr'] = None  # Heeft geen fiets

df_fiets_inkoop_feit = df_fiets_inkoop.copy()
df_fiets_inkoop_feit = df_fiets_inkoop_feit.rename(
    columns={'fiets': 'fietsnr'})
df_fiets_inkoop_feit['accessoirenummer'] = None  # Heeft geen accessoire

feit_inkoop = pd.concat(
    [df_acc_inkoop_feit, df_fiets_inkoop_feit]).reset_index(drop=True)
feit_inkoop['inkoop_id'] = feit_inkoop.index + 1  # Nieuwe unieke DWH sleutel
# Maak maand_id: 2023 en 10 wordt 202310
feit_inkoop['maand_id'] = feit_inkoop['inkoopjaar'].astype(
    str) + feit_inkoop['inkoopmaand'].astype(str).str.zfill(2)
feit_inkoop['maand_id'] = feit_inkoop['maand_id'].astype(int)


In [13]:
feit_inkoop = pd.merge(
    feit_inkoop, dim_fiets[['fietsnr', 'inkoopprijs']], on='fietsnr', how='left')
feit_inkoop = pd.merge(feit_inkoop, dim_accessoire[['accessoirenr', 'inkoopprijs']],
                       left_on='accessoirenummer', right_on='accessoirenr', how='left', suffixes=('_fiets', '_acc'))
feit_inkoop['inkoopprijs_per_stuk'] = feit_inkoop['inkoopprijs_fiets'].fillna(
    feit_inkoop['inkoopprijs_acc'])
feit_inkoop['totale_inkoopkosten'] = feit_inkoop['aantal'] * \
    feit_inkoop['inkoopprijs_per_stuk']

feit_inkoop = feit_inkoop[['inkoop_id', 'maand_id', 'fietsnr',
                           'accessoirenummer', 'aantal', 'totale_inkoopkosten']]
logger.info(f"-> FEIT_INKOOP getransformeerd: {len(feit_inkoop)} rijen.")


2026-04-09 14:17:41.067 | INFO     | __main__:<module>:12 - -> FEIT_INKOOP getransformeerd: 100 rijen.


In [14]:
# FEIT VERKOOP
df_acc_verk_feit = df_acc_verk.copy()
df_acc_verk_feit = df_acc_verk_feit.rename(
    columns={'accessoire': 'accessoirenummer', 'klant': 'klantnr', 'monteur': 'monteurnr'})
df_acc_verk_feit['fietsnr'] = None

df_fiets_verk_feit = df_fiets_verk.copy()
df_fiets_verk_feit = df_fiets_verk_feit.rename(
    columns={'fiets': 'fietsnr', 'klant': 'klantnr', 'monteur': 'monteurnr'})
df_fiets_verk_feit['accessoirenummer'] = None

feit_verkoop = pd.concat(
    [df_acc_verk_feit, df_fiets_verk_feit]).reset_index(drop=True)
feit_verkoop['verkoop_id'] = feit_verkoop.index + 1  # Unieke sleutel
feit_verkoop['datum'] = pd.to_datetime(feit_verkoop['datum'])
feit_verkoop['datum_id'] = feit_verkoop['datum'].dt.strftime(
    '%Y%m%d').astype(int)


In [15]:
feit_verkoop = pd.merge(
    feit_verkoop, dim_fiets[['fietsnr', 'inkoopprijs']], on='fietsnr', how='left')
feit_verkoop = pd.merge(feit_verkoop, dim_accessoire[['accessoirenr', 'inkoopprijs']],
                        left_on='accessoirenummer', right_on='accessoirenr', how='left', suffixes=('_fiets', '_acc'))
feit_verkoop['kostprijs_per_stuk'] = feit_verkoop['inkoopprijs_fiets'].fillna(
    feit_verkoop['inkoopprijs_acc'])
feit_verkoop['omzet'] = feit_verkoop['verkoopprijs']
feit_verkoop['winst'] = feit_verkoop['omzet'] - \
    (feit_verkoop['aantal'] * feit_verkoop['kostprijs_per_stuk'])

feit_verkoop = feit_verkoop[['verkoop_id', 'datum_id', 'klantnr', 'monteurnr',
                             'fietsnr', 'accessoirenummer', 'verkoopprijs', 'omzet', 'winst']]
logger.info(f"-> FEIT_VERKOOP getransformeerd: {len(feit_verkoop)} rijen.")


2026-04-09 14:17:41.110 | INFO     | __main__:<module>:13 - -> FEIT_VERKOOP getransformeerd: 250 rijen.


In [16]:
# FEIT ONDERHOUD
feit_onderhoud = df_ond.copy()
feit_onderhoud = feit_onderhoud.rename(
    columns={'fiets': 'fietsnr', 'monteur': 'monteurnr'})
feit_onderhoud['datum'] = pd.to_datetime(feit_onderhoud['datum'])
feit_onderhoud['datum_id'] = feit_onderhoud['datum'].dt.strftime(
    '%Y%m%d').astype(int)


In [17]:
feit_onderhoud['start_dt'] = pd.to_datetime(
    feit_onderhoud['starttijd'].astype(str).str[:5], format='%H:%M', errors='coerce')
feit_onderhoud['eind_dt'] = pd.to_datetime(
    feit_onderhoud['eindtijd'].astype(str).str[:5], format='%H:%M', errors='coerce')
feit_onderhoud['duur'] = (feit_onderhoud['eind_dt'] -
                          feit_onderhoud['start_dt']).dt.total_seconds() / 3600.0

feit_onderhoud = feit_onderhoud[['onderhoudnr', 'datum_id',
                                 'fietsnr', 'monteurnr', 'starttijd', 'eindtijd', 'duur']]
feit_onderhoud = feit_onderhoud.rename(columns={'onderhoudnr': 'onderhoud_id'})
logger.info(f"-> FEIT_ONDERHOUD getransformeerd: {len(feit_onderhoud)} rijen.")


2026-04-09 14:17:41.138 | INFO     | __main__:<module>:11 - -> FEIT_ONDERHOUD getransformeerd: 50 rijen.


In [18]:
# Datum tabel opruimen
dim_datum = dim_datum.drop(columns=['datum'])
logger.success("Stap 2: Transform afgerond.")


2026-04-09 14:17:41.150 | SUCCESS  | __main__:<module>:3 - Stap 2: Transform afgerond.


 # Stap 3 L (Load)

In [19]:
dwh_db = 'fietsen_dwh.db'
logger.info(f"Stap 3: Load - Verbinden met target DWH '{dwh_db}'...")

dwh_conn = sqlite3.connect(dwh_db)
cursor = dwh_conn.cursor()

# 1. Zet Foreign Keys constraint aan in SQLite
cursor.execute("PRAGMA foreign_keys = ON;")

try:
    # 2. Laad de Dimensies
    logger.info("Laden van Dimensies (met indexering)...")

    dim_datum.to_sql('DIM_DATUM', dwh_conn, if_exists='replace', index=False)
    cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_datum ON DIM_DATUM(datum_id);")

    dim_klant.to_sql('DIM_KLANT', dwh_conn, if_exists='replace', index=False)
    cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_klant ON DIM_KLANT(klantnr);")

    dim_monteur.to_sql('DIM_MONTEUR', dwh_conn, if_exists='replace', index=False)
    cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_monteur ON DIM_MONTEUR(monteurnr);")

    dim_fiets.to_sql('DIM_FIETS', dwh_conn, if_exists='replace', index=False)
    cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_fiets ON DIM_FIETS(fietsnr);")

    dim_accessoire.to_sql('DIM_ACCESSOIRE', dwh_conn, if_exists='replace', index=False)
    cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_acc ON DIM_ACCESSOIRE(accessoirenr);")
    logger.success("Dimensies succesvol geladen.")


    # 3. Maak de Feitentabellen expliciet aan MET Foreign Keys
    logger.info("Aanmaken en laden van Feitentabellen (met Foreign Keys)...")

    # FEIT: VERKOOP
    cursor.execute("DROP TABLE IF EXISTS FEIT_VERKOOP;")
    cursor.execute("""
    CREATE TABLE FEIT_VERKOOP (
        verkoop_id INTEGER PRIMARY KEY,
        datum_id INTEGER,
        klantnr INTEGER,
        monteurnr INTEGER,
        fietsnr INTEGER,
        accessoirenummer INTEGER,
        verkoopprijs REAL,
        omzet REAL,
        winst REAL,
        FOREIGN KEY(datum_id) REFERENCES DIM_DATUM(datum_id),
        FOREIGN KEY(klantnr) REFERENCES DIM_KLANT(klantnr),
        FOREIGN KEY(monteurnr) REFERENCES DIM_MONTEUR(monteurnr),
        FOREIGN KEY(fietsnr) REFERENCES DIM_FIETS(fietsnr),
        FOREIGN KEY(accessoirenummer) REFERENCES DIM_ACCESSOIRE(accessoirenr)
    );
    """)
    feit_verkoop.to_sql('FEIT_VERKOOP', dwh_conn, if_exists='append', index=False)
    logger.info("-> FEIT_VERKOOP weggeschreven.")


    # FEIT: ONDERHOUD
    cursor.execute("DROP TABLE IF EXISTS FEIT_ONDERHOUD;")
    cursor.execute("""
    CREATE TABLE FEIT_ONDERHOUD (
        onderhoud_id INTEGER PRIMARY KEY,
        datum_id INTEGER,
        fietsnr INTEGER,
        monteurnr INTEGER,
        starttijd TEXT,
        eindtijd TEXT,
        duur REAL,
        FOREIGN KEY(datum_id) REFERENCES DIM_DATUM(datum_id),
        FOREIGN KEY(fietsnr) REFERENCES DIM_FIETS(fietsnr),
        FOREIGN KEY(monteurnr) REFERENCES DIM_MONTEUR(monteurnr)
    );
    """)
    feit_onderhoud.to_sql('FEIT_ONDERHOUD', dwh_conn, if_exists='append', index=False)
    logger.info("-> FEIT_ONDERHOUD weggeschreven.")


    # FEIT: INKOOP
    cursor.execute("DROP TABLE IF EXISTS FEIT_INKOOP;")
    cursor.execute("""
    CREATE TABLE FEIT_INKOOP (
        inkoop_id INTEGER PRIMARY KEY,
        maand_id INTEGER,
        fietsnr INTEGER,
        accessoirenummer INTEGER,
        aantal INTEGER,
        totale_inkoopkosten REAL,
        FOREIGN KEY(fietsnr) REFERENCES DIM_FIETS(fietsnr),
        FOREIGN KEY(accessoirenummer) REFERENCES DIM_ACCESSOIRE(accessoirenr)
    );
    """)
    feit_inkoop.to_sql('FEIT_INKOOP', dwh_conn, if_exists='append', index=False)
    logger.info("-> FEIT_INKOOP weggeschreven.")

    # Sla op
    dwh_conn.commit()
    logger.success("Stap 3: Load afgerond. Data Warehouse is up-to-date!")

except Exception as e:
    dwh_conn.rollback()
    logger.error(f"Fout tijdens het laden van het DWH: {e}")

finally:
    dwh_conn.close()
    logger.info("🏁 Database verbinding gesloten. ETL script voltooid.")

2026-04-09 14:17:41.164 | INFO     | __main__:<module>:2 - Stap 3: Load - Verbinden met target DWH 'fietsen_dwh.db'...
2026-04-09 14:17:41.166 | INFO     | __main__:<module>:12 - Laden van Dimensies (met indexering)...
2026-04-09 14:17:41.194 | ERROR    | __main__:<module>:101 - Fout tijdens het laden van het DWH: Execution failed on sql 'DROP TABLE "DIM_DATUM"': FOREIGN KEY constraint failed
2026-04-09 14:17:41.195 | INFO     | __main__:<module>:105 - 🏁 Database verbinding gesloten. ETL script voltooid.
